# 03 · Join Sofascore + Capology — England Premier League 23/24

Integración de estadísticas de rendimiento (Sofascore) con datos salariales (Capology)
para la temporada **2023/24 de Premier League inglesa**.

**Flujo de matching:**
1. Normalización de nombres (tildes, mayúsculas, caracteres especiales)
2. `TEAM_MAP`: alineación manual de nombres de equipo entre fuentes
3. Merge exacto normalizado
4. Fuzzy matching en cuatro niveles:
   - Score ≥ 0.90 → aceptación automática
   - 0.75 ≤ score < 0.90 → revisión manual
   - 0.50 ≤ score < 0.75 → revisión manual estricta
   - score < 0.50 → revisión manual muy estricta
5. Revisión de jugadores sin salario
6. Guardado en `data/master/`

---

## 1. Imports y rutas

In [1]:
import pandas as pd
import unicodedata
import re
from rapidfuzz import fuzz
from pathlib import Path
from IPython.display import display

ROOT       = Path.cwd().parents[1]
SF_DIR     = ROOT / 'data' / 'processed' / 'sofascore'
CG_DIR     = ROOT / 'data' / 'processed' / 'capology'
MASTER_DIR = ROOT / 'data' / 'master'
MASTER_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Rutas configuradas')
print(f'   Root:   {ROOT}')
print(f'   Master: {MASTER_DIR}')

✅ Rutas configuradas
   Root:   d:\USER\Desktop\TFM
   Master: d:\USER\Desktop\TFM\data\master


## 2. Función de normalización

In [2]:
def normalize(s):
    """
    Normaliza un string para comparación: elimina tildes, pasa a minúsculas,
    elimina caracteres especiales y espacios extra.
    """
    if pd.isna(s):
        return ''
    s = str(s)
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    s = re.sub(r'[^a-z0-9\s]', ' ', s.lower().strip())
    return re.sub(r'\s+', ' ', s).strip()

print('✅ Función definida')

✅ Función definida


## 3. Carga de datos

In [3]:
df_sf = pd.read_csv(SF_DIR / 'df_england_2324.csv').copy()
df_cg = pd.read_csv(CG_DIR / 'cg_england_2324.csv').copy()

print(f'Sofascore:  {df_sf.shape[0]} jugadores | {df_sf.shape[1]} columnas')
print(f'Capology:   {df_cg.shape[0]} jugadores | {df_cg.shape[1]} columnas')

Sofascore:  570 jugadores | 116 columnas
Capology:   610 jugadores | 9 columnas


## 4. Normalización

In [4]:
df_sf['player_norm'] = df_sf['player'].apply(normalize)
df_sf['team_norm']   = df_sf['team'].apply(normalize)
df_cg['player_norm'] = df_cg['player'].apply(normalize)
df_cg['team_norm']   = df_cg['club'].apply(normalize)

print('✅ Normalización aplicada')

✅ Normalización aplicada


## 5. Alineación de equipos (TEAM_MAP)

### 5.1 Identificar discrepancias de nombres de equipo

In [5]:
solo_sf = set(df_sf['team_norm'].unique()) - set(df_cg['team_norm'].unique())
solo_cg = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())

print('En Sofascore pero no en Capology:')
for e in sorted(solo_sf): print(f'   {e}')
print()
print('En Capology pero no en Sofascore:')
for e in sorted(solo_cg): print(f'   {e}')

En Sofascore pero no en Capology:
   brighton hove albion
   newcastle united
   tottenham hotspur
   west ham united

En Capology pero no en Sofascore:
   brighton
   newcastle
   tottenham
   west ham


### 5.2 Aplicar TEAM_MAP

Rellenar con las discrepancias identificadas en la celda anterior.

In [6]:
# ── Ajustar según la celda anterior ──────────────────────────
TEAM_MAP = {'brighton':'brighton hove albion',
            'newcastle':'newcastle united',
            'tottenham':'tottenham hotspur',
            'west ham':'west ham united'

}
# ─────────────────────────────────────────────────────────────

df_cg['team_norm'] = df_cg['team_norm'].replace(TEAM_MAP)

diff = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())
if diff:
    print(f'⚠️  Equipos de CG aún sin match en SF: {diff}')
else:
    print('✅ Todos los equipos alineados')


✅ Todos los equipos alineados


## 6. Merge exacto normalizado

In [7]:
df_merged = df_sf.merge(
    df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
            'position', 'age', 'nationality']],
    on=['player_norm', 'team_norm'],
    how='left'
)

matched = df_merged['gross_annual_eur'].notna().sum()
total   = len(df_merged)

print(f'Merge exacto: {matched}/{total} ({matched/total:.1%})')
print(f'Sin emparejar: {total - matched}')

Merge exacto: 500/570 (87.7%)
Sin emparejar: 70


## 7. Fuzzy matching sobre los no emparejados

Se generan candidatos para todos los jugadores sin match exacto,
sin umbral mínimo, y se clasifican en cuatro niveles.

In [8]:
df_unmatched = df_merged[df_merged['gross_annual_eur'].isna()].copy()
cg_by_team   = df_cg.groupby('team_norm')['player_norm'].apply(list).to_dict()

rows = []
for _, row in df_unmatched[['player','team','player_norm','team_norm']].drop_duplicates().iterrows():
    candidates = cg_by_team.get(row['team_norm'], [])
    best_match, best_score = None, 0
    for cand in candidates:
        score = fuzz.ratio(row['player_norm'], cand) / 100
        if score > best_score:
            best_score = score
            best_match = cand
    if best_match is not None:
        rows.append({
            'player_sf'  : row['player'],
            'team'       : row['team'],
            'player_norm': row['player_norm'],
            'team_norm'  : row['team_norm'],
            'cg_match'   : best_match,
            'score'      : round(best_score, 3)
        })

df_candidates = pd.DataFrame(rows).sort_values('score', ascending=False)
auto_matches     = df_candidates[df_candidates['score'] >= 0.90].copy()
review_matches   = df_candidates[(df_candidates['score'] >= 0.75) & (df_candidates['score'] < 0.90)].copy()
low_matches      = df_candidates[(df_candidates['score'] >= 0.50) & (df_candidates['score'] < 0.75)].copy()
very_low_matches = df_candidates[df_candidates['score'] < 0.50].copy()

print(f'Auto-aceptados    (score ≥ 0.90):          {len(auto_matches)}')
print(f'Revisión media    (0.75 ≤ score < 0.90):   {len(review_matches)}')
print(f'Revisión estricta (0.50 ≤ score < 0.75):   {len(low_matches)}')
print(f'Revisión muy est. (score < 0.50):           {len(very_low_matches)}')

Auto-aceptados    (score ≥ 0.90):          5
Revisión media    (0.75 ≤ score < 0.90):   7
Revisión estricta (0.50 ≤ score < 0.75):   33
Revisión muy est. (score < 0.50):           25


### 7.1 Matches automáticos (score ≥ 0.90)

Revisar para confirmar que todos son correctos.

In [9]:
auto_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
2,Łukasz Fabiański,West Ham United,lukasz fabianski,0.968
46,Joshua Dasilva,Brentford,joshua da silva,0.966
11,Mykhaylo Mudryk,Chelsea,mykhailo mudryk,0.933
22,Yehor Yarmolyuk,Brentford,yegor yarmolyuk,0.933
6,Wesley Foderingham,Sheffield United,wes foderingham,0.909


### 7.2 Revisión media (0.75 ≤ score < 0.90)

Añadir a `EXCLUDE_FROM_FUZZY` el `player_norm` de los incorrectos.

In [10]:
review_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
20,Valentino Livramento,Newcastle United,tino livramento,0.857
9,Đorđe Petrović,Chelsea,djordje petrovic,0.857
16,Jóhann Guðmundsson,Burnley,johann berg gudmundsson,0.850
1,Ben Brereton Díaz,Sheffield United,ben brereton,0.828
13,Edward Nketiah,Arsenal,eddie nketiah,0.815
8,Stefan Ortega,Manchester City,stefan ortega moreno,0.788
7,Pape Matar Sarr,Tottenham Hotspur,pape sarr,0.750


In [11]:
# ── Falsos positivos a excluir del nivel medio ────────────────
EXCLUDE_FROM_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

review_accepted = review_matches[~review_matches['player_norm'].isin(EXCLUDE_FROM_FUZZY)]
print(f'Aceptados: {len(review_accepted)} | Excluidos: {len(EXCLUDE_FROM_FUZZY)}')


Aceptados: 7 | Excluidos: 0


### 7.3 Revisión estricta (0.50 ≤ score < 0.75)

Por defecto ninguno se acepta. Añadir a `ACCEPT_LOW_FUZZY` los correctos.

In [12]:
low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
63,Dominic Sadi,Bournemouth,dominic solanke,0.741
24,Mads Roerslev,Brentford,mads roerslev rasmussen,0.722
18,Emerson Royal,Tottenham Hotspur,emerson,0.700
45,Mason Burstow,Chelsea,malo gusto,0.696
28,Alex Murphy,Newcastle United,jacob murphy,0.696
14,Hannibal Mejbri,Manchester United,hannibal,0.696
19,Bobby Decordova-Reid,Fulham,bobby reid,0.667
53,Omari Kellyman,Aston Villa,youri tielemans,0.621
67,Jamie Donley,Tottenham Hotspur,james maddison,0.615
68,Alex Matos,Chelsea,andrey santos,0.609


In [13]:
# ── Matches de score bajo confirmados manualmente ─────────────
ACCEPT_LOW_FUZZY = ['mads roerslev',
                    'emerson royal',
                    'hannibal mejbri',
                    'bobby decordova reid',
                    'emerson palmieri',
                    'igor julio',
                    'thiago alcantara'

]
# ─────────────────────────────────────────────────────────────

low_accepted = low_matches[low_matches['player_norm'].isin(ACCEPT_LOW_FUZZY)]
print(f'Aceptados del nivel bajo: {len(low_accepted)}')


Aceptados del nivel bajo: 7


### 7.4 Revisión muy estricta (score < 0.50)

Por defecto ninguno se acepta. Añadir a `ACCEPT_VERY_LOW_FUZZY` los correctos.

In [14]:
very_low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
34,Benicio Boaitey,Brighton & Hove Albion,facundo buonanotte,0.485
69,Thomas Cannon,Everton,nathan patterson,0.483
38,Michael Olakigbe,Brentford,charlie goode,0.483
3,Josh Acheampong,Chelsea,moises caicedo,0.483
0,Aleksandar Mitrović,Fulham,alex iwobi,0.483
55,Antwoine Hackford,Sheffield United,benie traore,0.483
58,Joseph Johnson,Luton Town,andros townsend,0.483
23,Alfie Gilchrist,Chelsea,malo gusto,0.480
5,Aymeric Laporte,Manchester City,rico lewis,0.480
59,Daniel Gore,Manchester United,raphael varane,0.480


In [15]:
# ── Matches very low confirmados manualmente ──────────────────
ACCEPT_VERY_LOW_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

very_low_accepted = very_low_matches[very_low_matches['player_norm'].isin(ACCEPT_VERY_LOW_FUZZY)]
print(f'Aceptados del nivel very low: {len(very_low_accepted)}')


Aceptados del nivel very low: 0


### 7.5 Aplicar todos los fuzzy matches aceptados

In [16]:
all_fuzzy    = pd.concat([auto_matches, review_accepted, low_accepted, very_low_accepted], ignore_index=True)
fuzzy_lookup = dict(zip(all_fuzzy['player_norm'], all_fuzzy['cg_match']))

df_merged['player_norm_fuzzy'] = df_merged.apply(
    lambda r: fuzzy_lookup.get(r['player_norm'], r['player_norm'])
    if pd.isna(r['gross_annual_eur']) else r['player_norm'],
    axis=1
)

df_final = (
    df_merged
    .drop(columns=['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality'])
    .merge(
        df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
               'position', 'age', 'nationality']],
        left_on=['player_norm_fuzzy', 'team_norm'],
        right_on=['player_norm', 'team_norm'],
        how='left'
    )
    .drop(columns=['player_norm_y', 'player_norm_fuzzy'])
    .rename(columns={'player_norm_x': 'player_norm'})
)

matched_final = df_final['gross_annual_eur'].notna().sum()
print(f'Resultado final: {matched_final}/{len(df_final)} ({matched_final/len(df_final):.1%})')
print(f'Sin salario:     {len(df_final) - matched_final}')

Resultado final: 519/570 (91.1%)
Sin salario:     51


## 8. Revisión de jugadores sin salario

Ordenados por equipo y minutos jugados para identificar si alguno debería tener salario.

In [17]:
sin_salario = (
    df_final[df_final['gross_annual_eur'].isna()]
    [['player', 'team', 'minutesPlayed', 'appearances', 'goals', 'assists']]
    .sort_values(['team', 'minutesPlayed'], ascending=[True, False])
    .reset_index(drop=True)
)

pd.set_option('display.max_rows', None)
print(f'Total sin salario: {len(sin_salario)}')
display(sin_salario)
pd.reset_option('display.max_rows')

Total sin salario: 51


,player,team,minutesPlayed,appearances,goals,assists
0,Ethan Nwaneri,Arsenal,13,1,0,0
1,Omari Kellyman,Aston Villa,35,2,0,0
2,Kaine Kesler-Hayden,Aston Villa,10,3,0,0
3,Jaden Philogene-Bidace,Aston Villa,10,1,0,0
4,Finley Munroe,Aston Villa,8,1,0,0
5,Jaidon Anthony,Bournemouth,153,3,0,0
6,Dominic Sadi,Bournemouth,1,1,0,0
7,Michael Olakigbe,Brentford,100,8,0,0
8,Odeluga Offiah,Brighton & Hove Albion,146,4,0,0
9,Mark O'Mahony,Brighton & Hove Albion,69,3,0,0


### 8.1 Comparación manual por equipo

Para cada equipo con jugadores sin salario se muestra la plantilla completa de Capology
ordenada alfabéticamente por nombre normalizado, facilitando la detección visual de matches fallidos.

In [18]:
equipos_sin_salario = sin_salario['team'].unique()

for equipo in sorted(equipos_sin_salario):
    sf_jugadores = sin_salario[sin_salario['team'] == equipo][['player', 'minutesPlayed']].sort_values('player')

    equipo_norm  = normalize(equipo)
    cg_jugadores = (
        df_cg[df_cg['team_norm'] == equipo_norm][['player', 'player_norm']]
        .sort_values('player_norm')
        .reset_index(drop=True)
    )

    print(f'\n{"="*60}')
    print(f'  {equipo}  —  SF sin salario:')
    display(sf_jugadores.reset_index(drop=True))
    print(f'  CG plantilla completa:')
    display(cg_jugadores)


  Arsenal  —  SF sin salario:


,player,minutesPlayed
0,Ethan Nwaneri,13


  CG plantilla completa:


,player,player_norm
0,Aaron Ramsdale,aaron ramsdale
1,Ben White,ben white
2,Bukayo Saka,bukayo saka
3,Cédric Soares,cedric soares
4,David Raya,david raya
5,Declan Rice,declan rice
6,Eddie Nketiah,eddie nketiah
7,Emile Smith Rowe,emile smith rowe
8,Fábio Vieira,fabio vieira
9,Gabriel Jesus,gabriel jesus



  Aston Villa  —  SF sin salario:


...

,player,player_norm
0,Aaron Cresswell,aaron cresswell
1,Alphonse Areola,alphonse areola
2,Angelo Ogbonna,angelo ogbonna
3,Ben Johnson,ben johnson
4,Conor Coventry,conor coventry
5,Danny Ings,danny ings
6,Divin Mubama,divin mubama
7,Edson Álvarez,edson alvarez
8,Emerson,emerson
9,James Ward-Prowse,james ward prowse



  Wolverhampton  —  SF sin salario:


,player,minutesPlayed
0,Leon Chiwome,175
1,Nathan Fraser,210
2,Tawanda Chirewa,165


  CG plantilla completa:


,player,player_norm
0,Boubacar Traoré,boubacar traore
1,Bruno Jordão,bruno jordao
2,Craig Dawson,craig dawson
3,Daniel Bentley,daniel bentley
4,Daniel Podence,daniel podence
5,Enso González,enso gonzalez
6,Fábio Silva,fabio silva
7,Hee-chan Hwang,hee chan hwang
8,Hugo Bueno,hugo bueno
9,Jean-Ricner Bellegarde,jean ricner bellegarde


In [19]:
# ── Matches manuales (nombres muy distintos o traspasos invernales) ──
# Formato: (player_norm_sf, team_norm_sf): (player_norm_cg, team_norm_cg)
MANUAL_MATCHES = {

}
# ────────────────────────────────────────────────────────────────────
print(f'Matches manuales definidos: {len(MANUAL_MATCHES)}')


Matches manuales definidos: 0


In [20]:
# Aplicar matches manuales sobre los que siguen sin salario
for (p_sf, t_sf), (p_cg, t_cg) in MANUAL_MATCHES.items():
    mask = (df_final['player_norm'] == p_sf) & (df_final['team_norm'] == t_sf) & (df_final['gross_annual_eur'].isna())
    datos_cg = df_cg[(df_cg['player_norm'] == p_cg) & (df_cg['team_norm'] == t_cg)]
    if not datos_cg.empty and mask.any():
        for col in ['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality']:
            df_final.loc[mask, col] = datos_cg[col].values[0]
        print(f'✅ Match manual aplicado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')
    else:
        print(f'⚠️  No encontrado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')

matched_tras_manual = df_final['gross_annual_eur'].notna().sum()
print(f'\nTras matches manuales: {matched_tras_manual}/{len(df_final)} ({matched_tras_manual/len(df_final):.1%})')


Tras matches manuales: 519/570 (91.1%)


In [21]:
pd.reset_option('display.max_rows')

## 9. Guardado

Una vez revisado todo, se eliminan las columnas auxiliares y se guarda en `data/master/`.

In [22]:
df_final = df_final.drop(columns=['player_norm', 'team_norm'])

nombre_salida = 'master_england_2324.csv'
df_final.to_csv(MASTER_DIR / nombre_salida, index=False)

print(f'✅ Guardado: {nombre_salida}')
print(f'   Jugadores totales:  {len(df_final)}')
print(f'   Con salario:        {df_final["gross_annual_eur"].notna().sum()}')
print(f'   Sin salario (NaN):  {df_final["gross_annual_eur"].isna().sum()}')
print(f'   Columnas:           {df_final.shape[1]}')

✅ Guardado: master_england_2324.csv
   Jugadores totales:  570
   Con salario:        519
   Sin salario (NaN):  51
   Columnas:           121
